In [33]:
import glob, os, contextlib
from pathlib import Path
import numpy as np
import pandas as pd
import altair as alt
from ecostyles import EcoStyles

styles = EcoStyles(); styles.register_and_enable_theme()
W, H = 400, 300

OUT = Path("g7_debt_outputs"); (OUT / "data").mkdir(parents=True, exist_ok=True)

@contextlib.contextmanager
def in_dir(path):
    old = Path.cwd(); os.chdir(path)
    try: yield
    finally: os.chdir(old)

def save(chart, name):
    """Try the likely ecostyles call patterns; fall back to Altair's own export."""
    attempts = [
        lambda: styles.save(chart, str(OUT), name=name, svg=True),
        lambda: styles.save(chart, name=name, svg=True, path=str(OUT)),
        lambda: styles.save(chart, str(OUT / name), svg=True),
    ]
    last = None
    for i, fn in enumerate(attempts, 1):
        try:
            fn(); print(f"saved via ecostyles, attempt {i}: {name}"); return
        except TypeError as err:
            last = err
    print(f"ecostyles attempts failed ({last}); using Altair export")
    stem = OUT / name
    chart.save(f"{stem}.json"); chart.save(f"{stem}.svg"); chart.save(f"{stem}.png", scale_factor=2)

def dump(df, name):
    df.to_csv(OUT / "data" / f"{name}.csv")

# Data log: IMF.RES:WEO_2026_APR_VINTAGE, downloaded 20 Oct 2026. Projections = 2026 onward.

In [34]:
RAW = glob.glob("../Data/dataset_*WEO_2026_APR_VINTAGE*.csv")[0]
print(RAW)
CODES = ["GGXWDG_NGDP", "GGXWDN_NGDP", "GGXONLB_NGDP", "GGXCNL_NGDP",
         "GGR_NGDP", "GGX_NGDP", "NGDP_D", "NGDP_RPCH", "NGDPD"]

df = pd.read_csv(RAW, encoding="utf-8-sig")
df["code"] = df.SERIES_CODE.str.split(".").str[1]
years = [c for c in df.columns if c.isdigit()]
long = (df[df.code.isin(CODES)]
        .melt(id_vars=["COUNTRY", "code"], value_vars=years, var_name="year", value_name="v"))
long["year"] = long.year.astype(int)
long["v"] = pd.to_numeric(long.v, errors="coerce")
p = (long.pivot_table(index=["COUNTRY", "year"], columns="code", values="v")
         .reset_index().rename(columns={"COUNTRY": "country"})
         .sort_values(["country", "year"]).reset_index(drop=True))

p["pb"], p["ob"] = p.GGXONLB_NGDP, p.GGXCNL_NGDP
p["interest"] = p.pb - p.ob                                   # net interest, % GDP
p["g"] = (1 + p.NGDP_RPCH/100) * (1 + p.groupby("country")["NGDP_D"].pct_change()) - 1
p["debt_lag"] = p.groupby("country")["GGXWDG_NGDP"].shift(1)
p["pb_star"] = p.interest - p.g/(1 + p.g) * p.debt_lag        # debt-stabilising primary balance
p["gap"] = p.pb - p.pb_star
p["prim_spend"] = p.GGR_NGDP - p.pb

../Data/dataset_2026-10-20T09_42_49.749383602Z_DEFAULT_INTEGRATION_IMF.RES_WEO_2026_APR_VINTAGE_1.0.0.csv


In [35]:
pre, post = (2014, 2019), (2024, 2029)
wm = lambda col, a, b: p[p.year.between(a, b)].groupby("country")[col].mean()
dec = pd.DataFrame({
    "revenue":       wm("GGR_NGDP", *post) - wm("GGR_NGDP", *pre),
    "primary_spend": -(wm("prim_spend", *post) - wm("prim_spend", *pre)),
    "snowball":      -(wm("pb_star", *post) - wm("pb_star", *pre)),
})
dec["gap_change"] = dec.sum(axis=1)
print(dec.round(2))

w = p.pivot(index="year", columns="country", values="NGDPD")
share = w.div(w.sum(axis=1), axis=0)
gap = p.pivot(index="year", columns="country", values="gap")
weighted = (gap * share).sum(axis=1)

                revenue  primary_spend  snowball  gap_change
country                                                     
Canada             1.74          -3.96      1.80       -0.42
France            -1.30           0.49      0.25       -0.57
Germany            1.74          -6.36     -0.50       -5.13
Italy              0.73          -1.14      1.28        0.87
Japan              2.60          -1.60      3.04        4.04
United Kingdom     4.02          -3.45     -0.37        0.20
United States     -0.28          -0.75     -0.53       -1.56


In [38]:
d = dec.rename(columns={"revenue": "Revenue", "primary_spend": "Primary spending",
                        "snowball": "Interest-growth", "gap_change": "Total"})
comps = ["Primary spending", "Interest-growth", "Revenue"]
order = d.Total.sort_values(ascending=False).index.tolist()
longd = d.reset_index().melt(id_vars=["country", "Total"], value_vars=comps,
                             var_name="component", value_name="pp")
bars = alt.Chart(longd).mark_bar(height=16).encode(
    y=alt.Y("country:N", sort=order, title=None),
    x=alt.X("pp:Q", title="Percentage points of GDP",
            scale=alt.Scale(domain=[-7.5, 9]), axis=alt.Axis(values=list(range(-8, 7, 2)))),
    color=alt.Color("component:N", title=None, sort=comps,
                    legend=alt.Legend(orient="top", direction="horizontal")))
tot = d.reset_index()[["country", "Total"]]
dots = alt.Chart(tot).mark_point(filled=True, color="black", size=60).encode(
    y=alt.Y("country:N", sort=order), x="Total:Q")
lab = alt.Chart(tot).mark_text(align="left", fontSize=12, fontWeight="bold").encode(
    y=alt.Y("country:N", sort=order), x=alt.datum(6.8), text=alt.Text("Total:Q", format="+.1f"))
fig1 = (bars + dots + lab).properties(width=W, height=H,
        title="Change in G7 fiscal gaps, 2014–19 to 2024–29")
save(fig1, "g7_fig01_fiscal_gap_decomposition"); dump(dec, "g7_fig01_fiscal_gap_decomposition")
fig1

saved via ecostyles, attempt 1: g7_fig01_fiscal_gap_decomposition


alt.LayerChart(...)

In [16]:
rows = []
for name, (a, b) in {"2014–19": pre, "2024–29": post}.items():
    rows.append({"period": name, "measure": "GDP-weighted", "gap": weighted.loc[a:b].mean()})
    rows.append({"period": name, "measure": "Median country", "gap": gap.loc[a:b].mean().median()})
wm_df = pd.DataFrame(rows)
base = alt.Chart(wm_df).encode(
    x=alt.X("period:N", title=None, axis=alt.Axis(labelAngle=0)),
    xOffset=alt.XOffset("measure:N"),
    y=alt.Y("gap:Q", title="% of GDP", scale=alt.Scale(domain=[-1.2, 0.4])),
    color=alt.Color("measure:N", title=None, legend=alt.Legend(orient="top", direction="horizontal")))
txt = alt.Text("gap:Q", format="+.2f")
pos = base.mark_text(dy=-8, fontSize=11).encode(text=txt).transform_filter("datum.gap > 0")
neg = base.mark_text(dy=13, fontSize=11).encode(text=txt).transform_filter("datum.gap <= 0")
fig2 = (base.mark_bar() + pos + neg).properties(width=W, height=H,
        title="G7 fiscal gap: weighted average vs median country")
save(fig2, "g7_fig02_weighted_vs_median"); dump(wm_df.set_index("period"), "g7_fig02_weighted_vs_median")
fig2

saved via ecostyles, attempt 1: g7_fig02_weighted_vs_median


alt.LayerChart(...)

In [39]:
dA = p[p.year.between(2000, 2031)][["country", "year", "GGXWDG_NGDP"]].rename(columns={"GGXWDG_NGDP": "debt"})
xsA = alt.Scale(domain=[2000, 2039], zero=False)
axA = alt.Axis(format="d", values=[2000, 2010, 2020, 2030], grid=False)
proj = alt.Chart(pd.DataFrame({"a": [2026], "b": [2031]})).mark_rect(opacity=0.07, color="#94a3b8").encode(
    x=alt.X("a:Q", scale=xsA, title=None, axis=axA), x2="b:Q")
ln = alt.Chart(dA).mark_line(strokeWidth=1.8).encode(
    x=alt.X("year:Q", title=None, scale=xsA, axis=axA), y=alt.Y("debt:Q", title="% of GDP"),
    color=alt.Color("country:N", legend=None))
last = dA[dA.year == 2031].sort_values("debt").copy()
def spread(vals, gap_=11):                      # stops end-of-line labels colliding
    v = np.array(vals, float); out = v.copy()
    for i in range(1, len(out)): out[i] = max(out[i], out[i-1] + gap_)
    return out - (out.mean() - v.mean())
last["ylab"] = spread(last.debt.values)
lab3 = alt.Chart(last).mark_text(align="left", dx=4, fontSize=11).encode(
    x=alt.X("year:Q", scale=xsA), y="ylab:Q", text="country:N", color=alt.Color("country:N", legend=None))
fig3 = (proj + ln + lab3).properties(width=W, height=H, title="General government gross debt, G7, 2000–2031")
save(fig3, "g7_fig03_debt_paths")
dump(dA.pivot(index="year", columns="country", values="debt"), "g7_fig03_debt_paths")
fig3

saved via ecostyles, attempt 1: g7_fig03_debt_paths


alt.LayerChart(...)

In [46]:
def gapchart(country, label=None, start=2005):
    label = label or country
    s = p[(p.country == country) & p.year.between(start, 2031)][["year", "pb", "pb_star"]]
    s = s.rename(columns={"pb": "Primary balance", "pb_star": "Debt-stabilising primary balance"}
                ).melt("year", var_name="series", value_name="v")
    xs = alt.Scale(domain=[start, 2031], zero=False)
    ax = alt.Axis(format="d", values=[2005, 2010, 2015, 2020, 2025, 2030], grid=False)
    pr = alt.Chart(pd.DataFrame({"a": [2026], "b": [2031]})).mark_rect(opacity=0.07, color="#94a3b8").encode(
        x=alt.X("a:Q", scale=xs, title=None, axis=ax), x2="b:Q")
    zero = alt.Chart(pd.DataFrame({"y": [0]})).mark_rule(strokeDash=[1, 5], color="#94a3b8").encode(y="y:Q")
    line = alt.Chart(s).mark_line(strokeWidth=1.8).encode(
        x=alt.X("year:Q", title=None, scale=xs, axis=ax), y=alt.Y("v:Q", title="% of GDP"),
        color=alt.Color("series:N", title=None, legend=alt.Legend(orient="top", direction="vertical", labelLimit=300)))
    return (pr + zero + line).properties(width=W, height=H,
            title=f"{label}: primary balance vs debt-stabilising balance")

fig4 = gapchart("United Kingdom", label="UK")
save(fig4, "g7_fig04_uk_pb_vs_stabilising")
dump(p[p.country == "United Kingdom"].set_index("year")[["pb", "pb_star"]], "g7_fig04_uk_pb_vs_stabilising")
fig4

saved via ecostyles, attempt 1: g7_fig04_uk_pb_vs_stabilising


alt.LayerChart(...)

In [19]:
p["nd_lag"] = p.groupby("country")["GGXWDN_NGDP"].shift(1)
p["c_primary"]  = -p.pb
p["c_interest"] = p.interest
p["c_growth"]   = -p.g/(1 + p.g) * p.nd_lag
p["c_sfa"] = (p.GGXWDN_NGDP - p.nd_lag) - (p.c_primary + p.c_interest + p.c_growth)
hist = p[p.year.between(2020, 2025)].groupby("country")[["c_primary", "c_interest", "c_growth", "c_sfa"]].sum()
hist["Total"] = hist.sum(axis=1)
print(hist.round(1))

h = hist.rename(columns={"c_primary": "Primary deficit", "c_interest": "Net interest",
                         "c_growth": "Growth & inflation", "c_sfa": "Residual"})
comps5 = ["Primary deficit", "Net interest", "Growth & inflation", "Residual"]
order5 = h.Total.sort_values().index.tolist()
lg = h.reset_index().melt(id_vars=["country", "Total"], value_vars=comps5, var_name="driver", value_name="pp")
bars5 = alt.Chart(lg).mark_bar(height=16).encode(
    y=alt.Y("country:N", sort=order5, title=None),
    x=alt.X("pp:Q", title="Percentage points of GDP, 2020–25 cumulative",
            scale=alt.Scale(domain=[-42, 75]), axis=alt.Axis(values=list(range(-40, 61, 20)))),
    color=alt.Color("driver:N", title=None, sort=comps5,
                    legend=alt.Legend(orient="top", direction="horizontal", columns=2)))
t5 = h.reset_index()[["country", "Total"]]
dots5 = alt.Chart(t5).mark_point(filled=True, color="black", size=60).encode(
    y=alt.Y("country:N", sort=order5), x="Total:Q")
lbl5 = alt.Chart(t5).mark_text(align="left", fontSize=12, fontWeight="bold").encode(
    y=alt.Y("country:N", sort=order5), x=alt.datum(63), text=alt.Text("Total:Q", format="+.1f"))
fig5 = (bars5 + dots5 + lbl5).properties(width=W, height=H, title="What drove the rise in G7 net debt, 2019–25")
save(fig5, "g7_fig05_net_debt_drivers"); dump(hist, "g7_fig05_net_debt_drivers")
fig5

code            c_primary  c_interest  c_growth  c_sfa  Total
country                                                      
Canada               18.2        -0.8      -4.8  -11.2    1.5
France               26.6        10.0     -20.5    3.7   19.8
Germany              13.5         3.8     -10.5    0.6    7.3
Italy                18.8        21.1     -28.9   -6.3    4.7
Japan                22.6         2.1     -22.4  -14.0  -11.7
United Kingdom       28.7        13.8     -26.6    2.8   18.7
United States        34.6        17.3     -32.5   -4.3   15.1
saved via ecostyles, attempt 1: g7_fig05_net_debt_drivers


alt.LayerChart(...)

In [41]:
pre_pb = p[p.year.between(2017, 2019)].groupby("country")["pb"].mean()
dd = p[p.year == 2025].set_index("country")["GGXWDG_NGDP"] - p[p.year == 2019].set_index("country")["GGXWDG_NGDP"]
sc = pd.DataFrame({"pre_pb": pre_pb, "debt_change": dd}).reset_index()
sc["group"] = np.where(sc.country == "United Kingdom", "United Kingdom", "Other G7")
pts = alt.Chart(sc).mark_point(filled=True, size=70).encode(
    x=alt.X("pre_pb:Q", title="Average primary balance, 2017–19 (% of GDP)",
            scale=alt.Scale(domain=[-4, 3.5]), axis=alt.Axis(grid=False)),
    y=alt.Y("debt_change:Q", title="Change in gross debt, 2019–25 (pp of GDP)", scale=alt.Scale(domain=[-2, 26])),
    color=alt.Color("group:N", legend=None, sort=["United Kingdom", "Other G7"]))
b6 = alt.Chart(sc).encode(x="pre_pb:Q", y="debt_change:Q", text="country:N")
txr = b6.mark_text(align="left", dx=7, fontSize=11).transform_filter("datum.country != 'France'")
txl = b6.mark_text(align="right", dx=-7, fontSize=11).transform_filter("datum.country == 'France'")
fig6 = (pts + txr + txl).properties(width=W, height=H, title="Debt rise since 2019 vs pre-shock primary balance")
save(fig6, "g7_fig06_buffer_scatter"); dump(sc.set_index("country"), "g7_fig06_buffer_scatter")
fig6

saved via ecostyles, attempt 1: g7_fig06_buffer_scatter


alt.LayerChart(...)

In [42]:
e = (p[p.year == 2031].set_index("country")["pb"] - p[p.year == 2025].set_index("country")["pb"]).rename("swing").reset_index()
e["group"] = np.where(e.country == "United Kingdom", "United Kingdom", "Other G7")
oE = e.sort_values("swing", ascending=False).country.tolist()
xs7 = alt.Scale(domain=[-2.6, 5])
bar7 = alt.Chart(e).mark_bar(height=16).encode(
    y=alt.Y("country:N", sort=oE, title=None),
    x=alt.X("swing:Q", title="Percentage points of GDP", scale=xs7),
    color=alt.Color("group:N", legend=None, sort=["United Kingdom", "Other G7"]))
lab7 = alt.Chart(e).encode(y=alt.Y("country:N", sort=oE), x=alt.X("swing:Q", scale=xs7),
                           text=alt.Text("swing:Q", format="+.1f"))
lpos = lab7.mark_text(align="left", dx=5, fontSize=12, fontWeight="bold").transform_filter("datum.swing >= 0")
lneg = lab7.mark_text(align="right", dx=-5, fontSize=12, fontWeight="bold").transform_filter("datum.swing < 0")
fig7 = (bar7 + lpos + lneg).properties(width=W, height=H, title="Projected change in primary balance, 2025 to 2031")
save(fig7, "g7_fig07_projected_pb_swing"); dump(e.set_index("country"), "g7_fig07_projected_pb_swing")
fig7

saved via ecostyles, attempt 1: g7_fig07_projected_pb_swing


alt.LayerChart(...)

In [44]:
summary = pd.DataFrame({
    "gross_debt_2019": p[p.year == 2019].set_index("country")["GGXWDG_NGDP"],
    "gross_debt_2025": p[p.year == 2025].set_index("country")["GGXWDG_NGDP"],
    "gross_debt_2031": p[p.year == 2031].set_index("country")["GGXWDG_NGDP"],
    "pb_2017_19_avg": pre_pb,
    "pb_2025": p[p.year == 2025].set_index("country")["pb"],
    "pb_2031": p[p.year == 2031].set_index("country")["pb"],
}).round(2)
dump(summary, "g7_summary_table")
dump(p, "g7_panel_full")
print(sorted(os.listdir(OUT)))

['data', 'g7_fig01_fiscal_gap_decomposition.json', 'g7_fig01_fiscal_gap_decomposition.png', 'g7_fig01_fiscal_gap_decomposition.svg', 'g7_fig02_weighted_vs_median.json', 'g7_fig02_weighted_vs_median.png', 'g7_fig02_weighted_vs_median.svg', 'g7_fig03_debt_paths.json', 'g7_fig03_debt_paths.png', 'g7_fig03_debt_paths.svg', 'g7_fig04_uk_pb_vs_stabilising.json', 'g7_fig04_uk_pb_vs_stabilising.png', 'g7_fig04_uk_pb_vs_stabilising.svg', 'g7_fig05_net_debt_drivers.json', 'g7_fig05_net_debt_drivers.png', 'g7_fig05_net_debt_drivers.svg', 'g7_fig06_buffer_scatter.json', 'g7_fig06_buffer_scatter.png', 'g7_fig06_buffer_scatter.svg', 'g7_fig07_projected_pb_swing.json', 'g7_fig07_projected_pb_swing.png', 'g7_fig07_projected_pb_swing.svg']


In [48]:
# ---------- CELL 11: SHOCK / DRIFT / BASELINE DECOMPOSITION (net debt, 2007-25) ----------
# Net-debt identity columns (same as Cell 7; repeated here so this cell runs on its own)
p["nd_lag"] = p.groupby("country")["GGXWDN_NGDP"].shift(1)
p["c_growth"] = -p.g / (1 + p.g) * p.nd_lag
p["c_sfa"] = (p.GGXWDN_NGDP - p.nd_lag) - (-p.pb + p.interest + p.c_growth)

def decompose(start=2007, end=2025, base=(2003, 2007),
              windows=((2008, 2010), (2020, 2022)), covid_base=None):
    out = []
    for c, s in p.groupby("country"):
        s = s.set_index("year")
        b0 = s.loc[base[0]:base[1], "pb"].mean()                      # pre-shock primary-balance norm
        b1 = s.loc[covid_base[0]:covid_base[1], "pb"].mean() if covid_base else b0
        baseline = shock = drift = 0.0
        for y in range(start + 1, end + 1):
            b = b1 if y >= windows[1][0] else b0
            baseline += -b                                            # what the pre-shock norm alone would add
            dev = -(s.loc[y, "pb"] - b)                               # deviation from the norm (+ adds to debt)
            if any(lo <= y <= hi for lo, hi in windows):
                shock += dev                                          # inside a shock window
            else:
                drift += dev                                          # outside the windows
        snow = s.loc[start + 1:end, ["interest", "c_growth"]].sum().sum()
        sfa = s.loc[start + 1:end, "c_sfa"].sum()
        total = s.loc[end, "GGXWDN_NGDP"] - s.loc[start, "GGXWDN_NGDP"]
        out.append(dict(country=c, baseline=baseline, shocks=shock, drift=drift,
                        snowball=snow, residual=sfa, total=total))
    return pd.DataFrame(out).set_index("country")

sd = decompose()
print((sd.baseline + sd.shocks + sd.drift + sd.snowball + sd.residual - sd.total).abs().max())  # should be ~0
print(sd.round(1))

alts = {
    "main":                          {},
    "windows 2008-11 & 2020-23":     dict(windows=((2008, 2011), (2020, 2023))),
    "windows 2009-10 & 2020-21":     dict(windows=((2009, 2010), (2020, 2021))),
    "baseline 2000-07":              dict(base=(2000, 2007)),
    "COVID-era baseline = 2017-19":  dict(covid_base=(2017, 2019)),
}
sens_shock = pd.DataFrame({k: decompose(**kw).shocks for k, kw in alts.items()}).round(1)
sens_drift = pd.DataFrame({k: decompose(**kw).drift for k, kw in alts.items()}).round(1)
print(sens_shock); print(sens_drift)
dump(sd, "g7_shock_drift_decomposition"); dump(sens_shock, "g7_shock_sensitivity"); dump(sens_drift, "g7_drift_sensitivity")

# ---------- FIG 8 ----------
h8 = sd.rename(columns={"baseline": "Pre-shock norm", "snowball": "Interest-growth",
                        "residual": "Residual", "shocks": "Shocks", "drift": "Drift", "total": "Total"})
comps8 = ["Pre-shock norm", "Interest-growth", "Residual", "Shocks", "Drift"]
order8 = h8.Total.sort_values(ascending=False).index.tolist()
lg8 = h8.reset_index().melt(id_vars=["country", "Total"], value_vars=comps8, var_name="driver", value_name="pp")
bars8 = alt.Chart(lg8).mark_bar(height=16).encode(
    y=alt.Y("country:N", sort=order8, title=None),
    x=alt.X("pp:Q", title="Percentage points of GDP, 2008–25 cumulative",
            scale=alt.Scale(domain=[-90, 125]), axis=alt.Axis(values=list(range(-80, 81, 40)))),
    color=alt.Color("driver:N", title=None, sort=comps8,
                    legend=alt.Legend(orient="top", direction="horizontal", columns=3)))
t8 = h8.reset_index()[["country", "Total"]]
dots8 = alt.Chart(t8).mark_point(filled=True, color="black", size=60).encode(
    y=alt.Y("country:N", sort=order8), x="Total:Q")
lbl8 = alt.Chart(t8).mark_text(align="left", fontSize=12, fontWeight="bold").encode(
    y=alt.Y("country:N", sort=order8), x=alt.datum(100), text=alt.Text("Total:Q", format="+.1f"))
fig8 = (bars8 + dots8 + lbl8).properties(width=W, height=H, title="Shocks, drift and the pre-shock norm in G7 net debt, 2007–25")
save(fig8, "g7_fig08_shock_drift"); dump(sd, "g7_fig08_shock_drift")
fig8

1.4210854715202004e-14
                baseline  shocks  drift  snowball  residual  total
country                                                           
Canada             -40.4    33.6   34.9      -8.1     -32.0  -11.9
France              16.1    21.2   18.2      -4.3      -1.5   49.7
Germany             -0.5     9.2  -11.4      -6.8       3.5   -5.9
Italy              -23.1    22.4    5.3      28.5      -1.8   31.3
Japan               66.2    15.8   -2.4     -18.0     -18.6   43.0
United Kingdom      25.5    28.5   18.1     -17.7       1.4   55.9
United States       25.8    38.5   26.5     -19.9     -18.3   52.6
                main  windows 2008-11 & 2020-23  windows 2009-10 & 2020-21  \
country                                                                      
Canada          33.6                       40.8                       29.8   
France          21.2                       26.0                       19.3   
Germany          9.2                        9.9               

alt.LayerChart(...)

In [49]:
s = p[p.country == "United Kingdom"].set_index("year")
b = s.loc[2003:2007, "pb"].mean()
t = pd.DataFrame({
    "primary_balance": s.pb,
    "deviation_from_norm": -(s.pb - b),
    "net_interest": s.interest,
    "growth_effect": s.c_growth,
    "residual": s.c_sfa,
    "change_in_net_debt": s.GGXWDN_NGDP - s.nd_lag,
}).loc[2008:2025]
t["bucket"] = ["shock" if (2008 <= y <= 2010 or 2020 <= y <= 2022) else "drift" for y in t.index]
print(round(b, 2)); print(t.round(2))

-1.42
      primary_balance  deviation_from_norm  net_interest  growth_effect  \
year                                                                      
2008            -3.50                 2.09          1.54          -1.19   
2009            -8.51                 7.09          1.46           1.23   
2010            -6.72                 5.30          2.53          -2.02   
2011            -4.63                 3.21          2.80          -2.22   
2012            -5.62                 4.20          2.34          -2.19   
2013            -3.90                 2.49          1.39          -2.85   
2014            -3.65                 2.23          1.82          -3.52   
2015            -3.03                 1.61          1.51          -2.15   
2016            -1.67                 0.25          1.61          -3.02   
2017            -0.68                -0.74          1.81          -3.45   
2018            -0.56                -0.85          1.67          -2.62   
2019            -0.

In [51]:
# ---------- CELL 12: RETURN TO PRE-SHOCK NORM (primary balance minus 2003-07 average) ----------
BASE = (2003, 2007)
norm = p[p.year.between(*BASE)].groupby("country")["pb"].mean()
p["pb_norm"] = p.country.map(norm)
p["dev"] = p.pb - p.pb_norm                       # primary balance minus pre-shock norm, pp of GDP

def first_return(s, after):
    ys = s[(s.index > after) & (s >= 0)].index
    return int(ys[0]) if len(ys) else np.nan

rows = []
for c, g in p.groupby("country"):
    s = g.set_index("year")["dev"]
    rows.append(dict(country=c, norm=norm[c],
                     worst_gfc=s.loc[2008:2012].min(), back_after_gfc=first_return(s, 2010),
                     worst_covid=s.loc[2020:2022].min(), back_after_covid=first_return(s, 2022),
                     dev_2025=s.loc[2025], dev_2031=s.loc[2031]))
ret = pd.DataFrame(rows).set_index("country")
print(ret.round(1))
dump(ret, "g7_return_to_norm")

# ---------- FIG 9 ----------
d9 = p[p.year.between(2005, 2031)][["country", "year", "dev"]].copy()
xs9 = alt.Scale(domain=[2005, 2038], zero=False)
ax9 = alt.Axis(format="d", values=[2005, 2010, 2015, 2020, 2025, 2030], grid=False)
proj9 = alt.Chart(pd.DataFrame({"a": [2026], "b": [2031]})).mark_rect(opacity=0.07, color="#94a3b8").encode(
    x=alt.X("a:Q", scale=xs9, title=None, axis=ax9), x2="b:Q")
zero9 = alt.Chart(pd.DataFrame({"y": [0]})).mark_rule(strokeDash=[1, 5], color="#94a3b8").encode(y="y:Q")
ln9 = alt.Chart(d9).mark_line(strokeWidth=1.8).encode(
    x=alt.X("year:Q", title=None, scale=xs9, axis=ax9),
    y=alt.Y("dev:Q", title="pp of GDP", scale=alt.Scale(domain=[-14, 6])),
    color=alt.Color("country:N", legend=None))

def spread2(vals, gap_=0.9):                       # keeps end-of-line labels from colliding
    v = np.array(vals, float); out = v.copy()
    for i in range(1, len(out)): out[i] = max(out[i], out[i - 1] + gap_)
    return out - (out.mean() - v.mean())
last9 = d9[d9.year == 2031].sort_values("dev").copy()
last9["ylab"] = spread2(last9.dev.values)
lab_base = alt.Chart(last9).encode(
    x=alt.X("year:Q", scale=xs9), y="ylab:Q", text="country:N", color=alt.Color("country:N", legend=None))
lab9 = lab_base.mark_text(align="left", dx=4, fontSize=11).transform_filter("datum.country != 'United Kingdom'")
lab9uk = lab_base.mark_text(align="left", dx=4, fontSize=11, fontWeight="bold").transform_filter("datum.country == 'United Kingdom'")

fig9 = (proj9 + zero9 + ln9 + lab9 + lab9uk).properties(
    width=W, height=H, title="Primary balance relative to pre-crisis norm, G7")
save(fig9, "g7_fig09_return_to_norm"); dump(d9.pivot(index="year", columns="country", values="dev"), "g7_fig09_return_to_norm")
fig9

                norm  worst_gfc  back_after_gfc  worst_covid  \
country                                                        
Canada           2.2       -6.1             NaN        -12.7   
France          -0.9       -4.2          2030.0         -6.8   
Germany          0.0       -2.3          2011.0         -3.9   
Italy            1.3       -2.2          2012.0         -7.4   
Japan           -3.7       -4.9          2015.0         -4.7   
United Kingdom  -1.4       -7.1          2017.0        -10.4   
United States   -1.4       -9.9          2022.0        -10.6   

                back_after_covid  dev_2025  dev_2031  
country                                               
Canada                       NaN      -4.3      -3.1  
France                    2030.0      -2.2       1.3  
Germany                      NaN      -1.8      -2.2  
Italy                     2028.0      -0.8       0.3  
Japan                     2023.0       2.8       1.4  
United Kingdom            2026.0      

alt.LayerChart(...)

In [55]:
# ---------- CELL 13: THREE MORE CHARTS (needs Cells 0-1) ----------
def spread2(vals, gap_=0.9):                       # keeps end-of-line labels from colliding
    v = np.array(vals, float); out = v.copy()
    for i in range(1, len(out)): out[i] = max(out[i], out[i - 1] + gap_)
    return out - (out.mean() - v.mean())

# FIG 10: UK minus the median of the other six G7 countries, revenue and primary spending
yrs = p[p.year.between(2005, 2031)]
med = yrs[yrs.country != "United Kingdom"].groupby("year")[["GGR_NGDP", "prim_spend"]].median()   # median of the other six
uk_ = yrs[yrs.country == "United Kingdom"].set_index("year")[["GGR_NGDP", "prim_spend"]]
gap10 = (uk_ - med).rename(columns={"GGR_NGDP": "Revenue", "prim_spend": "Primary spending"})
d10 = gap10.reset_index().melt("year", var_name="kind", value_name="v")
xs10 = alt.Scale(domain=[2005, 2038], zero=False)
ax10 = alt.Axis(format="d", values=[2005, 2010, 2015, 2020, 2025, 2030], grid=False)
proj10 = alt.Chart(pd.DataFrame({"a": [2026], "b": [2031]})).mark_rect(opacity=0.07, color="#94a3b8").encode(
    x=alt.X("a:Q", scale=xs10, title=None, axis=ax10), x2="b:Q")
zero10 = alt.Chart(pd.DataFrame({"y": [0]})).mark_rule(strokeDash=[1, 5], color="#94a3b8").encode(y="y:Q")
ln10 = alt.Chart(d10).mark_line(strokeWidth=1.8).encode(
    x=alt.X("year:Q", title=None, scale=xs10, axis=ax10),
    y=alt.Y("v:Q", title="pp of GDP vs rest-of-G7 median", scale=alt.Scale(domain=[-9, 1])),
    color=alt.Color("kind:N", legend=None, sort=["Revenue", "Primary spending"]))
last10 = d10[d10.year == 2031].sort_values("v").copy()
last10["ylab"] = spread2(last10.v.values, gap_=0.8)
lab10 = alt.Chart(last10).mark_text(align="left", dx=4, fontSize=11).encode(
    x=alt.X("year:Q", scale=xs10), y="ylab:Q", text="kind:N",
    color=alt.Color("kind:N", legend=None, sort=["Revenue", "Primary spending"]))
fig10 = (proj10 + zero10 + ln10 + lab10).properties(width=W, height=H, title="UK relative to the rest of the G7, % of GDP")
save(fig10, "g7_fig10_uk_vs_rest_of_g7"); dump(gap10, "g7_fig10_uk_vs_rest_of_g7")
print(gap10.loc[[2007, 2019, 2025, 2031]].round(1))

# FIG 11: net interest bill by country
d11 = p[p.year.between(2005, 2031)][["country", "year", "interest"]].copy()
xs11 = alt.Scale(domain=[2005, 2038], zero=False)
ax11 = alt.Axis(format="d", values=[2005, 2010, 2015, 2020, 2025, 2030], grid=False)
proj11 = alt.Chart(pd.DataFrame({"a": [2026], "b": [2031]})).mark_rect(opacity=0.07, color="#94a3b8").encode(
    x=alt.X("a:Q", scale=xs11, title=None, axis=ax11), x2="b:Q")
ln11 = alt.Chart(d11).mark_line(strokeWidth=1.8).encode(
    x=alt.X("year:Q", title=None, scale=xs11, axis=ax11), y=alt.Y("interest:Q", title="% of GDP"),
    color=alt.Color("country:N", legend=None))
last11 = d11[d11.year == 2031].sort_values("interest").copy()
last11["ylab"] = spread2(last11.interest.values, gap_=0.28)
lb11 = alt.Chart(last11).encode(x=alt.X("year:Q", scale=xs11), y="ylab:Q", text="country:N", color=alt.Color("country:N", legend=None))
lab11 = lb11.mark_text(align="left", dx=4, fontSize=11).transform_filter("datum.country != 'United Kingdom'")
lab11uk = lb11.mark_text(align="left", dx=4, fontSize=11, fontWeight="bold").transform_filter("datum.country == 'United Kingdom'")
fig11 = (proj11 + ln11 + lab11 + lab11uk).properties(width=W, height=H, title="Net interest bill, G7")
save(fig11, "g7_fig11_net_interest"); dump(d11.pivot(index="year", columns="country", values="interest"), "g7_fig11_net_interest")

# FIG 12: distance from stabilising debt, three-year averages (2023-25 vs 2029-31 IMF projection)
gp = p.pivot(index="year", columns="country", values="gap")
g12 = pd.DataFrame({"2023–25": gp.loc[2023:2025].mean(), "2029–31 (IMF projection)": gp.loc[2029:2031].mean()})
o12 = g12["2029–31 (IMF projection)"].sort_values(ascending=False).index.tolist()
l12 = g12.reset_index().melt("country", var_name="period", value_name="gap_")
rules12 = g12.reset_index().rename(columns={"2023–25": "a", "2029–31 (IMF projection)": "b"})
xs12 = alt.Scale(domain=[-4, 8.5])
rl = alt.Chart(rules12).mark_rule(color="#94a3b8", strokeWidth=1.5).encode(
    y=alt.Y("country:N", sort=o12, title=None), x=alt.X("a:Q", title="Fiscal gap, pp of GDP", scale=xs12), x2="b:Q")
dt = alt.Chart(l12).mark_point(filled=True, size=70, opacity=1).encode(
    y=alt.Y("country:N", sort=o12), x=alt.X("gap_:Q", scale=xs12),
    color=alt.Color("period:N", title=None, sort=["2023–25", "2029–31 (IMF projection)"],
                    legend=alt.Legend(orient="top", direction="horizontal")))
z12 = alt.Chart(pd.DataFrame({"x": [0]})).mark_rule(strokeDash=[1, 5], color="#94a3b8").encode(x="x:Q")
fig12 = (z12 + rl + dt).properties(width=W, height=H, title="Distance from stabilising debt, now vs IMF baseline")
save(fig12, "g7_fig12_fiscal_gap_now_vs_2031"); dump(g12, "g7_fig12_fiscal_gap_now_vs_2031")
print(g12.round(2))
print(d11[d11.year.isin([2007, 2019, 2025, 2031])].pivot(index="year", columns="country", values="interest").round(1))
fig12

saved via ecostyles, attempt 1: g7_fig10_uk_vs_rest_of_g7
code  Revenue  Primary spending
year                           
2007     -6.3              -2.9
2019     -7.8              -5.8
2025     -6.8              -4.8
2031     -2.5              -3.5
saved via ecostyles, attempt 1: g7_fig11_net_interest
saved via ecostyles, attempt 1: g7_fig12_fiscal_gap_now_vs_2031
                2023–25  2029–31 (IMF projection)
country                                          
Canada             2.97                      2.06
France            -1.22                      0.13
Germany           -0.26                     -1.76
Italy              0.90                      1.07
Japan              7.42                      1.64
United Kingdom    -0.61                      1.56
United States     -1.04                     -2.68
country  Canada  France  Germany  Italy  Japan  United Kingdom  United States
year                                                                         
2007        0.5     2.5   

alt.LayerChart(...)

In [53]:
raw = pd.read_csv(RAW, encoding="utf-8-sig")
raw["code"] = raw.SERIES_CODE.str.split(".").str[1]
yrs_ = [c for c in raw.columns if c.isdigit()]
maxdiff, n = 0.0, 0
for code in CODES:
    r = raw[raw.code == code].set_index("COUNTRY")[yrs_].apply(pd.to_numeric, errors="coerce")
    for c in r.index:
        for y in range(2000, 2032):
            a = r.loc[c, str(y)]; b = p[(p.country == c) & (p.year == y)][code]
            b = b.iloc[0] if len(b) else np.nan
            if pd.notna(a) and pd.notna(b): maxdiff = max(maxdiff, abs(a - b)); n += 1
print(f"{n} values compared against the raw CSV; max abs difference = {maxdiff}")

2010 values compared against the raw CSV; max abs difference = 0.0
